# Autoencoder

## 0. Imports

In [1]:
import os
import sys

# Get the absolute path of the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)


In [2]:
import itertools
from dataclasses import dataclass
from datetime import datetime

import pandas as pd
from pathlib import Path
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization,
    Activation,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

import evaluation as evals
import plotter as plotter

## 1. Load Data

In [3]:
data_dir = Path("../../data/")

# read features
features_path = data_dir / "features_df.csv"
features_df = pd.read_csv(f"{features_path}")
print(f"Features Shape: {features_df.shape}")

Features Shape: (1932, 110)


In [4]:
# optionally read the edgelist in for now
edgelist_path = data_dir / "edgelists" / "edgelist_full.csv"
edgelist = pd.read_csv(f"{edgelist_path}")
print(f"Edgelist Shape: {edgelist.shape}")

Edgelist Shape: (3730692, 3)


Some quick validations:

- Number of users in each?

In [5]:
print(
    f"Unique user count in features_df vs. edgelist: \
    {features_df.user_id.nunique()} \
    {edgelist.user_anchor.nunique()}"
)
feature_df_users = set(features_df.user_id.tolist())
edgelist_users = set(edgelist.user_anchor.tolist())
print(f"Intersection length = {len(feature_df_users.intersection(edgelist_users))}")

Unique user count in features_df vs. edgelist:     1932     1932
Intersection length = 1932


## 2. Pre-processing

In [6]:
# save user ids for mapping
user_ids = features_df["user_id"].values

# one-hot encode all categorical
cat_cols = features_df.select_dtypes(include=["object", "bool"]).columns.tolist()
if "user_id" in cat_cols:
    cat_cols.remove("user_id")
df_numeric = features_df.drop(columns=["user_id"])
df_numeric = pd.get_dummies(df_numeric, columns=cat_cols)
print("One-hot encoding complete")

# fillna with median
df_numeric = df_numeric.fillna(df_numeric.median())
print("Filled missing with median")

# standardize and construct the X matrix
scaler = StandardScaler()
X = scaler.fit_transform(df_numeric)
print(X.shape)

X_train, X_test, ids_train, ids_test = train_test_split(
    X, user_ids, test_size=0.2, random_state=42
)

print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

One-hot encoding complete
Filled missing with median
(1932, 116)
Train Shape: (1545, 116), Test Shape: (387, 116)


/var/folders/jh/9_qy7nd96v9_q0_x1sffyq9c0000gn/T/ipykernel_45811/2416148417.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = features_df.select_dtypes(include=["object", "bool"]).columns.tolist()


## 3. Training Autoencoder

### Autoencoder Factory

Here's a general class implementation for the autoencoder that we can reuse

In [7]:
@dataclass
class Config:
    encoding_dim: int
    layers: list[int]
    learning_rate: float
    dropout_rate: float
    patience: int
    bottleneck_activation: str
    batch_size: int

    def __str__(self) -> str:
        parts = [
            f"model_enc{self.encoding_dim}",
            f"dep{len(self.layers)}",
            f"lr{self.learning_rate}",
            f"pat{self.patience}",
            f"drop{self.dropout_rate}",
            f"batch{self.batch_size}",
            f"bn_act-{self.bottleneck_activation}",
        ]
        return "_".join(parts)


class AutoencoderBuilder:
    """Factory for creating autoencoders"""

    def __init__(self, input_dim, config: list[Config]):
        self.input_dim = input_dim
        self.config = config
        self.model = self._build_model()

    def _build_model(self):
        inputs = Input(shape=(self.input_dim,))

        # ENCODER
        x = inputs
        for units in self.config.layers:
            x = Dense(units)(x)
            x = BatchNormalization()(x)
            x = Activation("relu")(x)
            x = Dropout(self.config.dropout_rate)(x)

        # Bottleneck (latent space)
        x = Dense(self.config.encoding_dim)(x)
        x = BatchNormalization()(x)
        encoded = Activation(
            self.config.bottleneck_activation, name="bottleneck_output"
        )(x)

        # DECODER
        x = encoded
        for units in reversed(self.config.layers):
            x = Dense(units)(x)
            x = BatchNormalization()(x)
            x = Activation("relu")(x)
            x = Dropout(self.config.dropout_rate)(x)

        # Final reconstruction – no bottleneck or anything
        decoded = Dense(self.input_dim, activation="linear")(x)

        # Assemble
        autoencoder = Model(inputs, decoded)
        optimizer = Adam(learning_rate=self.config.learning_rate)
        autoencoder.compile(optimizer=optimizer, loss="mse")

        return autoencoder

    def get_encoder(self):
        """This creates a new model that stops at the 'encoded' layer
        We grab the layer by name or by index (0 is input, then dense, bn, act...)
        Finding by name is safest"""
        encoder_output = self.model.get_layer("bottleneck_output").output
        return Model(inputs=self.model.input, outputs=encoder_output)

### Hyperparameters

Below parameters don't need to be tuned

In [8]:
loss = "mse"
epochs = 100
batch_size = 32
validation_split = 0.2

These parameters can be tuned. I put them inside a `Config` class:

```python
@dataclass
class Config:
    encoding_dim: int
    layers: list[int]
    learning_rate: float
    dropout_rate: float
    patience: int
```

Learning Rate, Patience, and Dropout are regularizers, so more important if we find
a good model that isn't generalizing. The other two – encoding dimension and layers,
are critical to find a model structure that works

Here's the things we can basically change:

| Hyperparameter | What it does | Suggested Experiment | 
| --- | --- | ---
| Encoding Dim | The size of the "bottleneck." | If reconstruction loss is high, the bottleneck might be too tight. Try bumping 32 up to 48 or 64. |
| Depth | Adding more layers. | Try adding another layer (e.g., 128→64→32) to help the model learn more complex hierarchical features. | 

### Training Loop

In [ ]:
# The configuration grid
grid = {
    "encoding_dim": [32, 48, 64, 96],
    "layers": [[64], [256, 128], [512, 256, 128]],
    "learning_rate": [0.001, 0.0025, 0.005],
    "dropout_rate": [0.1, 0.2],
    "patience": [10, 20],
    "batch_size": [32, 256],
    "bottleneck_activation": ["relu", "linear"],
}

# generates all combinations
keys = grid.keys()
values = grid.values()
all_configs = [Config(**dict(zip(keys, v))) for v in itertools.product(*values)]

# NOTE: important bit that filters out invalid configurations
# suppose encoding_dim = 96, but I had only one layer with 64 neurons
# the problem is my encoder reduces input_dim -> 64 -> 96...that second transformation
# is redundant
valid_configs = []
for config in all_configs:
    # The last layer in the encoder sequence
    last_hidden_layer_size = config.layers[-1]

    # Only keep it if it shrinks into the bottleneck
    if config.encoding_dim < last_hidden_layer_size:
        valid_configs.append(config)

print(f"Generated {len(all_configs)} total combinations.")
print(f"Filtered down to {len(valid_configs)} valid funnel architectures.")

# testing for now
# valid_configs = [
#     Config(
#         encoding_dim=48,
#         layers=[64],
#         learning_rate=0.0025,
#         patience=10,
#         dropout_rate=0.1,
#     ),
#     Config(
#         encoding_dim=32,
#         layers=[512, 256, 128],
#         learning_rate=0.0025,
#         patience=10,
#         dropout_rate=0.1,
#     ),
#     # Config(
#     #     encoding_dim=96,
#     #     layers=[256, 128],
#     #     learning_rate=0.0010,
#     #     patience=10,
#     #     dropout_rate=0.1,
#     # ),
# ]
# print(f"Generated {len(valid_configs)} configurations for testing.")

In [ ]:


# for saving outputs
run_id = datetime.now().strftime("%Y%m%d_%H%M")
os.makedirs(f"experiments/{run_id}", exist_ok=True)

results = []

for i, config in enumerate(valid_configs):
    print(f"Running experiment {i + 1}/{len(valid_configs)}")

    # SETUP
    builder = AutoencoderBuilder(input_dim=X.shape[1], config=config)

    # Early Stopping
    early_stop = EarlyStopping(
        monitor="val_loss", patience=config.patience, restore_best_weights=True
    )

    # LR Scheduler
    # If loss plateaus for 3 epochs, multiply LR by 0.5
    lr_schedule = ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6
    )

    # TRAIN
    history = builder.model.fit(
        X_train,
        X_train,
        epochs=30,
        batch_size=config.batch_size,
        validation_split=0.2,
        verbose=0,
    )

    # SAVE
    model_name = f"model_enc{config.encoding_dim}_dep{len(config.layers)}_lr{config.learning_rate}_pat{config.patience}_drop{config.dropout_rate}"
    model_path = f"experiments/{run_id}/{model_name}.keras"
    builder.model.save(model_path)

    # EVALUATION

    # get validation MSE
    val_mse = history.history["val_loss"][-1]

    # get test set MSE
    test_mse = builder.model.evaluate(X_test, X_test, verbose=0)

    # get the encoder, and run our metrics
    encoder = builder.get_encoder()
    embeddings = encoder.predict(X_test, verbose=0)

    # evaluator
    evaluator = evals.RecommenderEvaluator(embeddings, ids_test, edgelist)
    metrics = evaluator.get_all_metrics()

    # plot
    # plotter.plot_score_distribution(
    #     evaluator,
    #     model_name=f"enc={config.encoding_dim}_dep={len(config.layers)}_lr={config.learning_rate}",
    # )

    # Flatten config into a dict for the DataFrame
    res_dict = {
        "encoding_dim": config.encoding_dim,
        "depth": len(config.layers),
        "lr": config.learning_rate,
        "patience": config.patience,
        "dropout_rate": config.dropout_rate,
        "val_mse": val_mse,
        "test_mse": test_mse,
        **metrics,
        "model_path": model_path,
    }
    results.append(res_dict)

# Analyze
results_df = pd.DataFrame(results).sort_values(by="val_mse")
results_df.to_csv(f"experiments/{run_id}/manifest.csv", index=False)

#### Recovery


In [ ]:
valid_configs = [
    ## Option 1
    Config(
        encoding_dim=32,
        layers=[64],
        learning_rate=0.0025,
        patience=10,
        dropout_rate=0.1,
        batch_size=32,
        bottleneck_activation="relu",
    ),
    Config(
        encoding_dim=32,
        layers=[64],
        learning_rate=0.0025,
        patience=10,
        dropout_rate=0.1,
        batch_size=32,
        bottleneck_activation="linear",
    ),
    Config(
        encoding_dim=32,
        layers=[64],
        learning_rate=0.0025,
        patience=10,
        dropout_rate=0.1,
        batch_size=256,
        bottleneck_activation="relu",
    ),
    Config(
        encoding_dim=32,
        layers=[64],
        learning_rate=0.0025,
        patience=10,
        dropout_rate=0.1,
        batch_size=256,
        bottleneck_activation="linear",
    ),
    ## Option 2
    Config(
        encoding_dim=48,
        layers=[64],
        learning_rate=0.005,
        patience=20,
        dropout_rate=0.1,
        batch_size=32,
        bottleneck_activation="relu",
    ),
    Config(
        encoding_dim=48,
        layers=[64],
        learning_rate=0.005,
        patience=20,
        dropout_rate=0.1,
        batch_size=32,
        bottleneck_activation="linear",
    ),
    Config(
        encoding_dim=48,
        layers=[64],
        learning_rate=0.005,
        patience=20,
        dropout_rate=0.1,
        batch_size=256,
        bottleneck_activation="relu",
    ),
    Config(
        encoding_dim=48,
        layers=[64],
        learning_rate=0.005,
        patience=20,
        dropout_rate=0.1,
        batch_size=256,
        bottleneck_activation="linear",
    ),
    ## Option 3
    Config(
        encoding_dim=96,
        layers=[512, 256, 128],
        learning_rate=0.005,
        patience=10,
        dropout_rate=0.2,
        batch_size=32,
        bottleneck_activation="relu",
    ),
    Config(
        encoding_dim=96,
        layers=[512, 256, 128],
        learning_rate=0.005,
        patience=10,
        dropout_rate=0.2,
        batch_size=32,
        bottleneck_activation="linear",
    ),
    Config(
        encoding_dim=96,
        layers=[512, 256, 128],
        learning_rate=0.005,
        patience=10,
        dropout_rate=0.2,
        batch_size=256,
        bottleneck_activation="relu",
    ),
    Config(
        encoding_dim=96,
        layers=[512, 256, 128],
        learning_rate=0.005,
        patience=10,
        dropout_rate=0.2,
        batch_size=256,
        bottleneck_activation="linear",
    ),
]
print(f"Generated {len(valid_configs)} configurations for testing.")


# for saving outputs
run_id = datetime.now().strftime("%Y%m%d_%H%M")
os.makedirs(f"experiments/{run_id}", exist_ok=True)

results = []

for i, config in enumerate(valid_configs):
    print(f"Running experiment {i + 1}/{len(valid_configs)}")

    # SETUP
    builder = AutoencoderBuilder(input_dim=X.shape[1], config=config)

    # Early Stopping
    early_stop = EarlyStopping(
        monitor="val_loss", patience=config.patience, restore_best_weights=True
    )

    # LR Scheduler
    # If loss plateaus for 3 epochs, multiply LR by 0.5
    lr_schedule = ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6
    )

    # TRAIN
    history = builder.model.fit(
        X_train,
        X_train,
        epochs=30,
        batch_size=config.batch_size,
        validation_split=0.2,
        verbose=0,
    )

    # SAVE
    model_name = str(config)
    print(model_name)
    model_path = f"experiments/{run_id}/{model_name}.keras"
    builder.model.save(model_path)

    # EVALUATION

    # get validation MSE
    val_mse = history.history["val_loss"][-1]

    # get test set MSE
    test_mse = builder.model.evaluate(X_test, X_test, verbose=0)

    # get the encoder, and run our metrics
    encoder = builder.get_encoder()
    embeddings = encoder.predict(X_test, verbose=0)

    # evaluator
    evaluator = evals.RecommenderEvaluator(embeddings, ids_test, edgelist)
    metrics = evaluator.get_all_metrics()

    # plot
    # plotter.plot_score_distribution(
    #     evaluator,
    #     model_name=f"enc={config.encoding_dim}_dep={len(config.layers)}_lr={config.learning_rate}",
    # )

    # Flatten config into a dict for the DataFrame
    res_dict = {
        "encoding_dim": config.encoding_dim,
        "depth": len(config.layers),
        "lr": config.learning_rate,
        "patience": config.patience,
        "dropout_rate": config.dropout_rate,
        "batch_size": config.batch_size,
        "activation": config.bottleneck_activation,
        "val_mse": val_mse,
        "test_mse": test_mse,
        **metrics,
        "model_path": model_path,
    }
    results.append(res_dict)

# Analyze
results_df = pd.DataFrame(results)
results_df.to_csv(f"experiments/{run_id}/manifest.csv", index=False)


Generated 12 configurations for testing.
Running experiment 1/12
model_enc32_dep1_lr0.0025_pat10_drop0.1_batch32_bn_act-relu
Running experiment 2/12
model_enc32_dep1_lr0.0025_pat10_drop0.1_batch32_bn_act-linear
Running experiment 3/12
model_enc32_dep1_lr0.0025_pat10_drop0.1_batch256_bn_act-relu
Running experiment 4/12
model_enc32_dep1_lr0.0025_pat10_drop0.1_batch256_bn_act-linear
Running experiment 5/12
model_enc48_dep1_lr0.005_pat20_drop0.1_batch32_bn_act-relu
Running experiment 6/12
model_enc48_dep1_lr0.005_pat20_drop0.1_batch32_bn_act-linear
Running experiment 7/12
model_enc48_dep1_lr0.005_pat20_drop0.1_batch256_bn_act-relu
Running experiment 8/12
model_enc48_dep1_lr0.005_pat20_drop0.1_batch256_bn_act-linear
Running experiment 9/12
model_enc96_dep3_lr0.005_pat10_drop0.2_batch32_bn_act-relu
Running experiment 10/12
model_enc96_dep3_lr0.005_pat10_drop0.2_batch32_bn_act-linear
Running experiment 11/12
model_enc96_dep3_lr0.005_pat10_drop0.2_batch256_bn_act-relu
Running experiment 12/12


# Analysis

This part is to understand what features are truly influencing our model in meaningful
ways

In [ ]:
# Create a lookup for the test users from the original dataframe
# setting user_id as index makes reindexing trivial
feature_lookup = features_df.set_index("user_id")

# Re-retrieve the features in the EXACT order of ids_test
test_meta = feature_lookup.loc[ids_test].reset_index()

In [ ]:
encoder = builder.get_encoder()
all_embeddings = encoder.predict(X, verbose=0)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from umap import UMAP

# 1. Run UMAP (if you haven't yet)
reducer = UMAP(n_neighbors=20, min_dist=0.25, random_state=42)
embedding_2d = reducer.fit_transform(all_embeddings)

In [ ]:
# 2. Pick an artist feature to visualize
# Options: 'artist_entropy', 'hipster_gap', 'nunique_artist', 'artist_concentration_index'
target_feature = "temporal_total_sessions"
color_values = feature_lookup[target_feature]

# 3. Plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    embedding_2d[:, 0],
    embedding_2d[:, 1],
    c=color_values,
    cmap="viridis",  # 'magma' or 'plasma' are also great for entropy
    s=5,
    alpha=0.7,
)

plt.colorbar(scatter, label=target_feature)
plt.title(f"UMAP of User Latent Space colored by {target_feature}")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
sns.despine()
plt.show()


In [ ]:
# Create a DataFrame of your 32 dimensions
latent_cols = [f"dim_{i}" for i in range(embeddings.shape[1])]
latent_df = pd.DataFrame(embeddings, columns=latent_cols)

# Pick the artist features
# artist_features = [
#     "temporal_total_sessions",
#     "temporal_avg_session_track_count",
#     "temporal_avg_artist_diversity_ratio",
#     "temporal_avg_energy_trajectory",
#     "temporal_avg_session_hour",
# ]
artist_features = [
    "loyal_track_count",
    "early_loyal_listens",
    "early_loyal_ratio",
    # "user_type_loyal",
    "genre_unique_count",
    "genre_entropy",
]
original_artist_data = test_meta[artist_features]

# Calculate correlation between the 32 dimensions and the artist features
corr_matrix = pd.concat([latent_df, original_artist_data], axis=1).corr()
artist_latent_corr = corr_matrix.loc[latent_cols, artist_features]

# Plot heatmap
plt.figure(figsize=(10, 12))
sns.heatmap(artist_latent_corr, annot=True, cmap="coolwarm", center=0)
plt.title("Correlation: Latent Dimensions vs. Artist Features")
plt.show()


The `user_embeddings` is the key output of our autoencoder – there are 1932 rows, 
each representing one user. Each user has 32 columns – which is the vector representing
our users. To generate "good matches", we can look at cosine similarity of the users
based on this vector. 

# OLD STUFF

Below this are diffierent architectures Roxana tried. We can replace it once we 
incorporate into hyperparameter training

### second try

In [ ]:
input_dim = X.shape[1]
encoding_dim = 64  # larger than first model to retain more info

input_layer = Input(shape=(input_dim,))
# Encoder
x = Dense(128, activation="relu")(input_layer)
x = BatchNormalization()(x)  # added batch normalization
x = Dropout(0.1)(x)  # added dropout
encoded = Dense(encoding_dim, activation="relu")(x)

# Decoder
x = Dense(128, activation="relu")(encoded)
x = BatchNormalization()(x)
decoded = Dense(input_dim, activation="linear")(x)

autoencoder = Model(input_layer, decoded)
encoder = Model(input_layer, encoded)

autoencoder.compile(optimizer="adam", loss="mae")  # changed to MAE might be more r

# increase epochs
autoencoder.fit(X, X, epochs=100, batch_size=64, validation_split=0.1, verbose=0)

# Evaluate
print("Extracting embeddings and calculating similarity...")
user_embeddings = encoder.predict(X)
sim_matrix = cosine_similarity(user_embeddings)

sim_df = pd.DataFrame(sim_matrix, index=user_ids, columns=user_ids)

# new evaluation
k = 10
match_generated_for = sim_df.index.nunique()
universe_size = edgelist.user_1.nunique()
print(f"Similarity DF user count: {match_generated_for}")
print(f"Edgelist DF user count: {universe_size}")
per_user, summary = evals.evaluate_topk(
    similarity_df,
    edgelist,
    k=k,
    anchor_col="user_1",
    other_col="user_2",
    score_col="similarity",
)
good_matches = per_user.loc[per_user.intersection > 0, :].shape[0]
print("Total count of users we found a 'good' match:", good_matches)
print(
    f"Pct users we found a 'good' match: {(good_matches / match_generated_for) * 100:.2f}%"
)
# for k, v in summary.items():
#     print(f"{k}: {v:.2f}")


# ensure we're getting users in both df
# valid_edgelist = edgelist[
#     edgelist["user_id_anchor"].isin(user_ids)
#     & edgelist["user_id_positive"].isin(user_ids)
# ]

# print(f"Testing on {len(valid_edgelist)} pairs...")

# # get top 10 match from encoder
# hits = 0
# for anchor in valid_edgelist["user_id_anchor"].unique():
#     # get positives of this anchor in edgelist
#     positives = set(
#         valid_edgelist[valid_edgelist["user_id_anchor"] == anchor]["user_id_positive"]
#     )

#     # get this anchor's top 11 similarity (cuz self might be top 1)
#     top_n = sim_df.loc[anchor].nlargest(11).index.tolist()
#     if anchor in top_n:
#         top_n.remove(anchor)
#     top_10 = set(top_n[:10])

#     # any positive hit would count as one hit
#     if len(positives.intersection(top_10)) > 0:
#         hits += 1

# final_score = hits / len(valid_edgelist["user_id_anchor"].unique())
# print(f"New User-level Recall@10: {final_score:.2%}")

In [ ]:
per_user.groupby("intersection")["user_id"].nunique()

In [ ]:
# Pair level recall：
user_embeddings = encoder.predict(X)
sim_matrix = cosine_similarity(user_embeddings)
similarity_df = pd.DataFrame(sim_matrix, index=user_ids, columns=user_ids)

valid_edgelist = edgelist[
    edgelist["user_id_anchor"].isin(user_ids)
    & edgelist["user_id_positive"].isin(user_ids)
]

print(f"Starting Pair-level Evaluation on {len(valid_edgelist)} pairs...")

hits = 0
total_pairs = len(valid_edgelist)

for idx, row in valid_edgelist.iterrows():
    anchor = row["user_id_anchor"]
    positive = row["user_id_positive"]

    top_10_similar = similarity_df.loc[anchor].drop(anchor).nlargest(10).index.tolist()

    if positive in top_10_similar:
        hits += 1

pair_level_recall = hits / total_pairs

print("\n[Final Results - Pair-level]")
print(
    f"Deep Autoencoder Reconstruction Loss (MAE): {autoencoder.evaluate(X, X, verbose=0):.4f}"
)
print(f"Pair-level Recall@10: {pair_level_recall:.2%}")
print(f"Successfully found {hits} out of {total_pairs} preset pairs.")

### third try

In [ ]:
from sklearn.preprocessing import MinMaxScaler  # 换成了 MinMaxScaler

# changed to use min max scaler
scaler = MinMaxScaler()
X = scaler.fit_transform(df_numeric)

# 2. deeper Autoencoder
input_dim = X.shape[1]
encoding_dim = 64  # Bottleneck dim

input_layer = Input(shape=(input_dim,))

# Input -> 256 -> 128 -> 64
# Encoder
encoder_layers = Sequential(
    [
        Dense(256, activation="relu"),
        BatchNormalization(),
        Dense(128, activation="relu"),
        BatchNormalization(),
        Dense(encoding_dim, activation="relu"),  # Latent Space
    ]
)

# Decoder
decoder_layers = Sequential(
    [
        Dense(128, activation="relu"),
        BatchNormalization(),
        Dense(256, activation="relu"),
        BatchNormalization(),
        Dense(
            input_dim, activation="sigmoid"
        ),  # cuz input is now 0-1，so use sigmoid for output
    ]
)

encoded_repr = encoder_layers(input_layer)
decoded_repr = decoder_layers(encoded_repr)

autoencoder = Model(inputs=input_layer, outputs=decoded_repr)
encoder_model = Model(inputs=input_layer, outputs=encoded_repr)

autoencoder.compile(optimizer="adam", loss="mae")  # MAE

# increased epochs
autoencoder.fit(X, X, epochs=100, batch_size=64, validation_split=0.1, verbose=1)

# eval
user_embeddings = encoder_model.predict(X)
sim_matrix = cosine_similarity(user_embeddings)
similarity_df = pd.DataFrame(sim_matrix, index=user_ids, columns=user_ids)

valid_edgelist = edgelist[
    edgelist["user_id_anchor"].isin(user_ids)
    & edgelist["user_id_positive"].isin(user_ids)
]

hits = 0
unique_anchors = valid_edgelist["user_id_anchor"].unique()

for anchor in unique_anchors:
    positives = set(
        valid_edgelist[valid_edgelist["user_id_anchor"] == anchor]["user_id_positive"]
    )

    top_10 = similarity_df.loc[anchor].drop(anchor).nlargest(10).index.tolist()

    if any(p in top_10 for p in positives):
        hits += 1

recall_at_10 = hits / len(unique_anchors)
print("\n[Final Results]")
print(
    f"Deep Autoencoder Reconstruction Loss (MAE): {autoencoder.evaluate(X, X, verbose=0):.4f}"
)
print(f"User-level Recall@10: {recall_at_10:.2%}")
print(f"Matched {hits} users out of {len(unique_anchors)} total anchors in edgelist.")

In [ ]:
# Pair level recall：
user_embeddings = encoder_model.predict(X)
sim_matrix = cosine_similarity(user_embeddings)
similarity_df = pd.DataFrame(sim_matrix, index=user_ids, columns=user_ids)

valid_edgelist = edgelist[
    edgelist["user_id_anchor"].isin(user_ids)
    & edgelist["user_id_positive"].isin(user_ids)
]

print(f"Starting Pair-level Evaluation on {len(valid_edgelist)} pairs...")

hits = 0
total_pairs = len(valid_edgelist)

for idx, row in valid_edgelist.iterrows():
    anchor = row["user_id_anchor"]
    positive = row["user_id_positive"]

    top_10_similar = similarity_df.loc[anchor].drop(anchor).nlargest(10).index.tolist()

    if positive in top_10_similar:
        hits += 1

pair_level_recall = hits / total_pairs

print("\n[Final Results - Pair-level]")
print(
    f"Deep Autoencoder Reconstruction Loss (MAE): {autoencoder.evaluate(X, X, verbose=0):.4f}"
)
print(f"Pair-level Recall@10: {pair_level_recall:.2%}")
print(f"Successfully found {hits} out of {total_pairs} preset pairs.")